In [ ]:
# Produces Transects

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Point, MultiLineString
from shapely.ops import linemerge, unary_union
import os
import logging
import time
from datetime import timedelta

# =====================================================
# CONFIGURATION
# =====================================================
ROOT_DIR = r"C:\Users\KyleSteen.AzureAD\Documents\Splitting_ROW_every_100m\Iowa"

HW_GPKG             = os.path.join(ROOT_DIR, "NHS_Iowa_dissolve.gpkg")
ROW_GPKG            = os.path.join(ROOT_DIR, "Final_ROWs_Iowa_Erase_RP_multi_greater100sqm.gpkg")
TRANSECT_LINES_GPKG = os.path.join(ROOT_DIR, "Iowa_100m_transect_lines.gpkg")

TRANSECT_HALF_LENGTH = 500   # meters each side
INTERVAL             = 100   # meters between transects

# =====================================================
# LOGGER
# =====================================================
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

def eta(start, done, total):
    if done == 0:
        return "estimating..."
    rate = done / (time.time() - start)
    return str(timedelta(seconds=int((total - done) / rate)))

start_time = time.time()
logger.info("Starting transect generation...")

# =====================================================
# LOAD DATA
# =====================================================
logger.info("Loading ROW polygons...")
rows = gpd.read_file(ROW_GPKG)
logger.info(f"  Loaded {len(rows):,} ROW polygons | CRS: {rows.crs}")

logger.info("Loading highway centerlines...")
hw_lines = gpd.read_file(HW_GPKG)

if hw_lines.crs != rows.crs:
    logger.info(f"  Reprojecting highways from {hw_lines.crs} → {rows.crs}")
    hw_lines = hw_lines.to_crs(rows.crs)

logger.info(f"  Loaded {len(hw_lines):,} highway segments")

# =====================================================
# DISSOLVE + MERGE INTO CONTINUOUS LINES
# =====================================================
logger.info("Dissolving and merging highway segments...")
hw_dissolved = hw_lines.dissolve()
merged = linemerge(unary_union(hw_dissolved.geometry.values))

if isinstance(merged, MultiLineString):
    all_parts = list(merged.geoms)
    # Filter out parts shorter than one interval — they're just topology noise
    lines_to_sample = [p for p in all_parts if p.length >= INTERVAL]
    dropped         = len(all_parts) - len(lines_to_sample)
    logger.info(f"  MultiLineString — {len(all_parts):,} parts after merge")
    logger.info(f"  Dropped {dropped:,} parts shorter than {INTERVAL}m")
    logger.info(f"  Sampling {len(lines_to_sample):,} parts")
    logger.info(f"  Longest:  {max(p.length for p in lines_to_sample)/1000:.1f} km")
    logger.info(f"  Median:   {sorted(p.length for p in lines_to_sample)[len(lines_to_sample)//2]/1000:.1f} km")
else:
    lines_to_sample = [merged]
    logger.info(f"  Single continuous LineString — {merged.length/1000:.1f} km")

# =====================================================
# GENERATE PERPENDICULAR TRANSECTS
# =====================================================
logger.info(f"Generating perpendicular transects every {INTERVAL}m...")
transect_geoms = []

hw_start    = time.time()
total_lines = len(lines_to_sample)

for i, line in enumerate(lines_to_sample, start=1):
    if line is None or line.is_empty:
        continue

    distances = np.arange(0, line.length + INTERVAL, INTERVAL)

    for dist in distances:
        dist  = min(dist, line.length)
        point = line.interpolate(dist)

        d0 = max(dist - 1e-4, 0)
        d1 = min(dist + 1e-4, line.length)
        tang_start = line.interpolate(d0)
        tang_end   = line.interpolate(d1)

        dx = tang_end.x - tang_start.x
        dy = tang_end.y - tang_start.y

        perp_dx = -dy
        perp_dy =  dx
        norm = np.hypot(perp_dx, perp_dy)
        if norm == 0:
            continue
        perp_dx /= norm
        perp_dy /= norm

        p1 = Point(point.x - perp_dx * TRANSECT_HALF_LENGTH,
                   point.y - perp_dy * TRANSECT_HALF_LENGTH)
        p2 = Point(point.x + perp_dx * TRANSECT_HALF_LENGTH,
                   point.y + perp_dy * TRANSECT_HALF_LENGTH)

        transect_geoms.append(LineString([p1, p2]))

    if i % 50 == 0 or i == total_lines:
        logger.info(
            f"  Lines {i:,}/{total_lines:,} ({i/total_lines:.1%}) "
            f"| Transects so far: {len(transect_geoms):,} "
            f"| ETA {eta(hw_start, i, total_lines)}"
        )

logger.info(f"Total transects generated: {len(transect_geoms):,}")

# =====================================================
# SAVE
# =====================================================
transect_gdf = gpd.GeoDataFrame(
    {"transect_id": range(len(transect_geoms))},
    geometry=transect_geoms,
    crs=rows.crs
)

logger.info(f"Writing transect lines → {TRANSECT_LINES_GPKG}")
transect_gdf.to_file(TRANSECT_LINES_GPKG, driver="GPKG")

logger.info(f"Done in {timedelta(seconds=int(time.time() - start_time))}")

In [1]:
# Clips Input ROW to Transects - claude



# Clips Input ROW to Transects - Optimized
# Improvements:
#   - STRtree spatial index for transect clipping (major speedup)
#   - Chunked unary_union for both ROW dissolve and line noding
#   - Progress logging on ALL loops (no more silent hangs)
#   - Batch intersection using geopandas overlay as fallback option
#   - Elapsed time reported at every major step

import geopandas as gpd
from shapely.ops import unary_union, polygonize
from shapely.strtree import STRtree
import os
import logging
import time
from datetime import timedelta

# =====================================================
# CONFIGURATION
# =====================================================
ROOT_DIR = r"C:\Users\KyleSteen.AzureAD\Documents\Splitting_ROW_every_100m\Iowa"

ROW_GPKG            = os.path.join(ROOT_DIR, "Final_ROWs_Iowa_Erase_RP_multi_greater100sqm.gpkg")
TRANSECT_LINES_GPKG = os.path.join(ROOT_DIR, "Iowa_100m_transect_lines.gpkg")
OUTPUT_GPKG         = os.path.join(ROOT_DIR, "Final_ROWs_Iowa_Erase_RP_multi_greater100sqm_100mtransects.gpkg")
# How many geometries to merge per chunk in unary_union calls.
# Lower = more progress updates but more overhead. 500-1000 is a good range.
UNION_CHUNK_SIZE = 1000

# =====================================================
# LOGGER
# =====================================================
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()
script_start = time.time()

def elapsed():
    return str(timedelta(seconds=int(time.time() - script_start)))

def chunked_union(geom_list, chunk_size=UNION_CHUNK_SIZE, label="geometries"):
    """
    Performs unary_union in chunks to avoid silent multi-hour hangs on large inputs.
    Merges chunk results in a second pass.
    """
    total = len(geom_list)
    if total == 0:
        return None
    if total <= chunk_size:
        return unary_union(geom_list)

    chunks = [geom_list[i:i + chunk_size] for i in range(0, total, chunk_size)]
    logger.info(f"  Chunked union: {total:,} {label} → {len(chunks):,} chunks of ~{chunk_size}")

    merged = []
    t0 = time.time()
    for idx, chunk in enumerate(chunks, 1):
        merged.append(unary_union(chunk))
        if idx % 10 == 0 or idx == len(chunks):
            elapsed_c = time.time() - t0
            rate = idx / elapsed_c if elapsed_c > 0 else 1
            eta = str(timedelta(seconds=int((len(chunks) - idx) / rate)))
            logger.info(f"    Chunk {idx:,}/{len(chunks):,} ({idx/len(chunks):.1%}) | ETA {eta}")

    logger.info(f"  Final merge of {len(merged):,} chunk results...")
    return unary_union(merged)


# =====================================================
# LOAD
# =====================================================
logger.info("=" * 60)
logger.info("Starting ROW splitting (optimized polygonize approach)")
logger.info("=" * 60)

logger.info("Loading ROW polygons...")
t0 = time.time()
rows = gpd.read_file(ROW_GPKG)
logger.info(f"  {len(rows):,} polygon(s) loaded | CRS: {rows.crs} | {timedelta(seconds=int(time.time()-t0))}")

logger.info("Loading transect lines...")
t0 = time.time()
transects = gpd.read_file(TRANSECT_LINES_GPKG)
if transects.crs != rows.crs:
    logger.info(f"  Reprojecting transects {transects.crs} → {rows.crs}")
    transects = transects.to_crs(rows.crs)
logger.info(f"  {len(transects):,} transects loaded | {timedelta(seconds=int(time.time()-t0))}")


# =====================================================
# STEP 1 — Dissolve ROW polygons (chunked)
# =====================================================
logger.info("-" * 60)
logger.info("STEP 1: Dissolving ROW polygons into single geometry (chunked)...")
t0 = time.time()

row_geom = chunked_union(list(rows.geometry.values), label="ROW polygons")

logger.info(f"  ROW union complete in {timedelta(seconds=int(time.time()-t0))}")
logger.info(f"  Geometry type: {row_geom.geom_type}")


# =====================================================
# STEP 2 — Build spatial index, clip transects to ROW
# =====================================================
logger.info("-" * 60)
logger.info("STEP 2: Clipping transects to ROW using spatial index...")
t0 = time.time()

# Build STRtree on individual ROW polygons for fast local lookup
logger.info("  Building STRtree spatial index on ROW polygons...")
row_geoms_arr = list(rows.geometry.values)
row_index = STRtree(row_geoms_arr)
logger.info("  Spatial index built")

clipped_lines = []
skipped = 0
total_t = len(transects)

for i, geom in enumerate(transects.geometry, start=1):
    # Query spatial index for candidate ROW polygons near this transect
    candidate_idxs = row_index.query(geom)

    if len(candidate_idxs) == 0:
        skipped += 1
        continue

    # Union only the local candidates (much faster than full row_geom)
    if len(candidate_idxs) == 1:
        local_geom = row_geoms_arr[candidate_idxs[0]]
    else:
        local_geom = unary_union([row_geoms_arr[j] for j in candidate_idxs])

    clipped = geom.intersection(local_geom)

    if clipped.is_empty:
        skipped += 1
        continue

    if clipped.geom_type in ("LineString", "MultiLineString"):
        clipped_lines.append(clipped)

    if i % 500 == 0 or i == total_t:
        elapsed_c = time.time() - t0
        rate = i / elapsed_c if elapsed_c > 0 else 1
        eta = str(timedelta(seconds=int((total_t - i) / rate)))
        logger.info(
            f"  {i:,}/{total_t:,} ({i/total_t:.1%})"
            f" | Kept: {len(clipped_lines):,}"
            f" | Skipped: {skipped:,}"
            f" | ETA {eta}"
        )

logger.info(f"  Clipping done in {timedelta(seconds=int(time.time()-t0))}")
logger.info(f"  {len(clipped_lines):,} transect segments kept | {skipped:,} skipped (outside ROW)")


# =====================================================
# STEP 3 — Build planar graph (chunked unary_union)
# =====================================================
logger.info("-" * 60)
logger.info("STEP 3: Building planar line graph (chunked noding)...")
t0 = time.time()

row_boundary = row_geom.boundary
all_lines = [row_boundary] + clipped_lines
logger.info(f"  Total lines to node: {len(all_lines):,} (1 boundary + {len(clipped_lines):,} transects)")

noded = chunked_union(all_lines, label="lines")

logger.info(f"  Noding complete in {timedelta(seconds=int(time.time()-t0))}")


# =====================================================
# STEP 4 — Polygonize
# =====================================================
logger.info("-" * 60)
logger.info("STEP 4: Polygonizing noded lines...")
t0 = time.time()

raw_polygons = list(polygonize(noded))
logger.info(f"  {len(raw_polygons):,} raw polygons produced in {timedelta(seconds=int(time.time()-t0))}")

if len(raw_polygons) == 0:
    logger.error("  !! No polygons produced — check that transects fully cross the ROW boundaries")
    raise RuntimeError("Polygonize returned 0 polygons. Aborting.")


# =====================================================
# STEP 5 — Clip polygons to ROW (with progress)
# =====================================================
logger.info("-" * 60)
logger.info("STEP 5: Clipping raw polygons to ROW extent...")
t0 = time.time()

final_pieces = []
total_raw = len(raw_polygons)

for idx, poly in enumerate(raw_polygons):
    clipped_poly = poly.intersection(row_geom)

    if clipped_poly.is_empty:
        continue

    if clipped_poly.geom_type == "MultiPolygon":
        final_pieces.extend(list(clipped_poly.geoms))
    elif clipped_poly.geom_type == "Polygon":
        final_pieces.append(clipped_poly)

    if idx % 5000 == 0 or idx == total_raw - 1:
        elapsed_c = time.time() - t0
        rate = (idx + 1) / elapsed_c if elapsed_c > 0 else 1
        eta = str(timedelta(seconds=int((total_raw - idx) / rate)))
        logger.info(
            f"  {idx+1:,}/{total_raw:,} ({(idx+1)/total_raw:.1%})"
            f" | Output so far: {len(final_pieces):,}"
            f" | ETA {eta}"
        )

logger.info(f"  Clipping done in {timedelta(seconds=int(time.time()-t0))}")
logger.info(f"  {len(final_pieces):,} final pieces")

if len(final_pieces) == 0:
    logger.error("  !! 0 final pieces after clipping — check geometry validity")
    raise RuntimeError("0 final pieces produced. Aborting before writing empty file.")


# =====================================================
# SAVE
# =====================================================
logger.info("-" * 60)
logger.info("Saving output...")
t0 = time.time()

out_gdf = gpd.GeoDataFrame(
    {"piece_id": range(len(final_pieces))},
    geometry=final_pieces,
    crs=rows.crs
)

logger.info(f"  Writing {len(out_gdf):,} features → {OUTPUT_GPKG}")
out_gdf.to_file(OUTPUT_GPKG, driver="GPKG")
logger.info(f"  Write complete in {timedelta(seconds=int(time.time()-t0))}")

logger.info("=" * 60)
logger.info(f"ALL DONE — Total elapsed: {elapsed()}")
logger.info("=" * 60)

[2026-03-27 09:46:52] ============================================================
[2026-03-27 09:46:52] Starting ROW splitting (optimized polygonize approach)
[2026-03-27 09:46:52] ============================================================
[2026-03-27 09:46:52] Loading ROW polygons...
[2026-03-27 09:46:53]   23,739 polygon(s) loaded | CRS: EPSG:5070 | 0:00:00
[2026-03-27 09:46:53] Loading transect lines...
[2026-03-27 09:46:53]   83,724 transects loaded | 0:00:00
[2026-03-27 09:46:53] ------------------------------------------------------------
[2026-03-27 09:46:53] STEP 1: Dissolving ROW polygons into single geometry (chunked)...
[2026-03-27 09:46:53]   Chunked union: 23,739 ROW polygons → 24 chunks of ~1000
[2026-03-27 09:46:56]     Chunk 10/24 (41.7%) | ETA 0:00:04
[2026-03-27 09:47:00]     Chunk 20/24 (83.3%) | ETA 0:00:01
[2026-03-27 09:47:01]     Chunk 24/24 (100.0%) | ETA 0:00:00
[2026-03-27 09:47:01]   Final merge of 24 chunk results...
[2026-03-27 09:47:14]   ROW union comp

In [4]:
import geopandas as gpd
gdf = gpd.read_file(r"C:\Users\KyleSteen.AzureAD\Documents\Splitting_ROW_every_100m\Iowa\Final_ROWs_Iowa_Erase_RP_multi_greater100sqm_100mtransects.gpkg")
print(gdf.geom_type.value_counts())
print(f"Total features: {len(gdf):,}")

Polygon    245494
Name: count, dtype: int64
Total features: 245,494


In [ ]:
# Creating Centerlines

import os
import arcpy
# =====================================================
# CONFIG
# =====================================================
arcpy.env.overwriteOutput = True

INPUT = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\ROW_Width_Splitting\Final_ROWs_Iowa.gdb\Final_ROWs_Iowa"
OUTPUT_GDB = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\ROW_Width_Splitting\ROW_Width_Splitting.gdb"

RASTER_NAME = "row_raster"
THIN_NAME   = "row_thin"
CENTERLINE  = os.path.join(OUTPUT_GDB, "ROW_Centerlines")

CELL_SIZE = 100  # meters (IMPORTANT — controls detail vs speed)

# =====================================================
# STEP 1 — Polygon → Raster
# =====================================================
print("Rasterizing polygon...")

row_raster = arcpy.conversion.PolygonToRaster(
    in_features=INPUT,
    value_field="OBJECTID",   # any field works
    out_rasterdataset=os.path.join(OUTPUT_GDB, RASTER_NAME),
    cell_assignment="MAXIMUM_AREA",
    cellsize=CELL_SIZE
)

# =====================================================
# STEP 2 — Thin (Skeletonize)
# =====================================================
print("Running thinning (skeletonization)...")

thin_raster = arcpy.sa.Thin(row_raster)

thin_raster.save(os.path.join(OUTPUT_GDB, THIN_NAME))

# =====================================================
# STEP 3 — Raster → Polyline
# =====================================================
print("Converting skeleton to polyline...")

arcpy.conversion.RasterToPolyline(
    in_raster=thin_raster,
    out_polyline_features=CENTERLINE,
    background_value="ZERO",
    simplify="SIMPLIFY"
)

# =====================================================
# STEP 4 — Optional Cleanup (recommended)
# =====================================================
print("Cleaning centerlines...")

# Remove tiny spurs
arcpy.management.DeleteIdentical(CENTERLINE, ["Shape"])

# Optional: dissolve into continuous lines
dissolved = os.path.join(OUTPUT_GDB, "ROW_Centerlines_Dissolved")
arcpy.management.Dissolve(CENTERLINE, dissolved)

print("Done.")